# 00 · Setup compartido

Este notebook **no se ejecuta solo**: cada notebook lo llama con `%run ./00_Setup`.

| Qué deja listo | Para qué |
|---|---|
| Esquema y volumen en Unity Catalog | Un lugar propio para el laboratorio (`lab_tvmaze_m34`) |
| Rutas y nombres de tablas | Un solo sitio donde cambiarlos |
| Esquema del JSON de TVMaze | Se define una vez y lo usa Auto Loader |
| Funciones del pipeline | `run_bronze()`, `build_silver()`, `build_gold()` y ayudantes |

```
API TVMaze ─► RAW (JSON en el Volumen) ─► BRONZE ─► SILVER ─► GOLD
                        Auto Loader ──┘   CDF+MERGE ┘  CDF+MERGE ┘
```

## 1. Ubicación: catálogo, esquema y volumen

In [0]:
import time, inspect, requests, json, re
import os
from databricks.sdk import WorkspaceClient
from datetime import date, timedelta,datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

USER = spark.sql("SELECT current_user()").first()[0]    # recibe las alertas del Job
LOG  = "pipeline_log"  

CATALOG = spark.sql("SELECT current_catalog()").first()[0]   # 'workspace' en la Free Edition
SCHEMA  = "proyecto_opensky"
VOLUME  = "datos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 2. Rutas y nombres de tablas

In [0]:
# Archivos (Volumen)
BASE            = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_PATH        = f"{BASE}/raw/flights"
SCHEMA_PATH     = f"{BASE}/autoloader/schema"       # Auto Loader guarda aquí el esquema
CHECKPOINT_PATH = f"{BASE}/autoloader/checkpoint"   # Auto Loader recuerda aquí qué archivos ya leyó

print(f"BASE: {BASE}")
print(f"RAW_PATH: {RAW_PATH}")
print(f"SCHEMA_PATH: {SCHEMA_PATH}")
print(f"CHECKPOINT_PATH: {CHECKPOINT_PATH}")

# Tablas (Delta)
BRONZE = "bronze_flights"
SILVER = "silver_vuelos"
FACT   = "gold_fact_vuelo"
DIM_AEROLINEA  = "gold_dim_aerolinea"
DIM_AEROPUERTO = "gold_dim_aeropuerto"
DIM_FECHA      = "gold_dim_fecha"
STATE  = "pipeline_state"

## 3. Esquema explícito del JSON

Solo los campos que usaremos. Lo que no está aquí, Auto Loader lo guarda en `_rescued_data`.

In [0]:
ESQUEMA_VUELOS = """
    icao24 STRING, firstSeen BIGINT, lastSeen BIGINT, callsign STRING,
    estDepartureAirport STRING, estArrivalAirport STRING,
    departureAirportCandidatesCount INT, arrivalAirportCandidatesCount INT
"""

## 4. Ayudantes: nombres, descarga de la API y versiones

In [0]:
from datetime import datetime, timezone

# Principales aeropuertos de Brasil (código ICAO)
AEROPUERTOS = [
    "SBGR",   # São Paulo – Guarulhos (internacional)
    "SBSP",   # São Paulo – Congonhas (nacional)
    "SBGL",   # Río de Janeiro – Galeão (internacional)
    "SBRJ",   # Río de Janeiro – Santos Dumont (nacional)
    "SBBR",   # Brasília
    "SBKP",   # Campinas – Viracopos
]
TIPOS       = ["arrival", "departure"]
OPENSKY_API = "https://opensky-network.org/api/flights"
ZONA_HORARIA = "America/Sao_Paulo"      # hora de Brasília (UTC-3), se usará en silver

CATALOGO_URL  = "https://davidmegginson.github.io/ourairports-data/airports.csv"
RAW_CATALOGOS = f"{BASE}/raw/catalogos"

def qualified_name(tabla):
    """Nombre completo de una tabla: catalogo.esquema.tabla"""
    return f"{CATALOG}.{SCHEMA}.{tabla}"


def table_version(tabla):
    """Última versión de una tabla Delta."""
    return spark.sql(f"DESCRIBE HISTORY {tabla}").agg(F.max("version")).first()[0]


def show_history(tabla):
    """Historial resumido de una tabla Delta."""
    display(spark.sql(f"DESCRIBE HISTORY {tabla}")
            .select("version", "timestamp", "operation", "operationMetrics")
            .orderBy("version"))

def opensky_headers():
    """Token OAuth2 con las credenciales del secret scope; sin ellas, acceso anónimo."""
    try:
        cid = dbutils.secrets.get("proyecto", "opensky_client_id")
        sec = dbutils.secrets.get("proyecto", "opensky_client_secret")
    except Exception:
        print("⚠ Sin credenciales: acceso anónimo")
        return {}
    r = requests.post("https://auth.opensky-network.org/auth/realms/opensky-network/"
                      "protocol/openid-connect/token",
                      data={"grant_type": "client_credentials", "client_id": cid, "client_secret": sec},
                      timeout=30)
    r.raise_for_status()
    return {"Authorization": f"Bearer {r.json()['access_token']}"}


def unix_dia(fecha):
    """Inicio y fin de un día en UTC, en segundos Unix (int)."""
    begin = int(datetime(fecha.year, fecha.month, fecha.day, tzinfo=timezone.utc).timestamp())
    return begin, begin + 86400

class LimiteAPI(Exception):
    """OpenSky respondió 429: se acabaron los créditos por ahora."""


def download_day(fecha, aeropuertos, headers, res):
    """Descarga un día (UTC) de llegadas y salidas de cada aeropuerto a RAW."""
    begin, end = unix_dia(fecha)
    for aeropuerto in aeropuertos:
        for tipo in TIPOS:
            destino = Path(RAW_PATH) / f"flights_{aeropuerto}_{tipo}_{fecha.isoformat()}.json"
            if destino.exists():
                res["existía"] += 1
                continue
            try:
                r = requests.get(f"{OPENSKY_API}/{tipo}", headers=headers, timeout=60,
                                 params={"airport": aeropuerto, "begin": begin, "end": end})
            except requests.RequestException as e:
                print(f"  ✘ {aeropuerto} {tipo} {fecha}: {e}")
                res["falló"] += 1
                continue

            if r.status_code == 429:                   # límite de la API: no tiene sentido seguir
                espera = r.headers.get("X-Rate-Limit-Retry-After-Seconds", "?")
                raise LimiteAPI(f"Límite de OpenSky alcanzado en {aeropuerto} {tipo} {fecha}. "
                                f"Reintenta en {espera} segundos.")
            if r.status_code == 404:                   # sin vuelos ese día
                res["sin vuelos"] += 1
                continue
            if r.status_code != 200:
                print(f"  ✘ {aeropuerto} {tipo} {fecha}: status {r.status_code}")
                res["falló"] += 1
                continue

            destino.parent.mkdir(parents=True, exist_ok=True)
            destino.write_bytes(r.content)             # JSON exactamente como lo devolvió la API
            res["nuevo"] += 1
            restantes = r.headers.get("X-Rate-Limit-Remaining")
            if restantes:
                res["créditos restantes"] = restantes
            time.sleep(1)


def download_days(inicio, dias, aeropuertos=None):
    """Descarga varios días seguidos; si OpenSky corta por límite, se detiene y guarda lo avanzado."""
    aeropuertos = aeropuertos or AEROPUERTOS
    headers = opensky_headers()
    res = {"nuevo": 0, "existía": 0, "sin vuelos": 0, "falló": 0}
    try:
        for i in range(dias):
            download_day(inicio + timedelta(days=i), aeropuertos, headers, res)
    except LimiteAPI as e:
        print(f"⏸ {e}")
        print("  Lo descargado quedó guardado. Vuelve a ejecutar más tarde: saltará lo que ya existe.")
    print(" | ".join(f"{k}: {v}" for k, v in res.items()))

def download_catalogos():
    """Descarga los catálogos públicos de OurAirports (aeropuertos y países) a RAW, tal cual llegan."""
    base_url = "https://davidmegginson.github.io/ourairports-data"
    for archivo in ("airports.csv", "countries.csv"):
        destino = Path(RAW_CATALOGOS) / archivo
        r = requests.get(f"{base_url}/{archivo}", timeout=120)
        r.raise_for_status()
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(r.content)
        print(f"Descargado: {destino} ({len(r.content)/1e6:.1f} MB)")


def raw_files():
    """Nombres de los archivos JSON que hay en RAW, ordenados."""
    if not Path(RAW_PATH).exists():
        return []
    return sorted(f.name for f in dbutils.fs.ls(RAW_PATH))

## 5. Auto Loader por lotes: RAW → bronze

Dos funciones separadas, como pide el diseño: una **lee** (detecta archivos nuevos) y otra **escribe** en bronze.
Corre por lotes: `trigger(availableNow=True)` procesa lo pendiente y se detiene.
La escritura es un **upsert**: cada lote se aplica con `MERGE` por `id` (inserta episodios nuevos, actualiza los que ya existen).

In [0]:
def create_bronze_table():
    """Crea bronze vacía con Change Data Feed activado (los cambios se registran desde la versión 0)."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {BRONZE} (
            {ESQUEMA_VUELOS},
            _rescued_data STRING, source_file STRING, ingested_at TIMESTAMP
        ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)


def autoloader_read():
    """LECTURA · Auto Loader detecta en RAW los archivos nuevos y los lee con el esquema explícito."""
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("multiLine", "true")                              # cada archivo es un arreglo JSON
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # lo que no encaje en el esquema va aquí
            .schema(ESQUEMA_VUELOS)
            .load(RAW_PATH)
            .select("*",
                    F.col("_metadata.file_name").alias("source_file"),
                    F.current_timestamp().alias("ingested_at")))


def upsert_to_bronze(lote, lote_id):
    """MERGE de un lote en bronze.
    Clave normal: (icao24, firstSeen).
    Vuelos sin hora de salida (firstSeen nulo): se identifican por (icao24, lastSeen)."""
    from delta.tables import DeltaTable
    con_salida = lote.where("firstSeen IS NOT NULL").dropDuplicates(["icao24", "firstSeen"])
    sin_salida = lote.where("firstSeen IS NULL").dropDuplicates(["icao24", "lastSeen"])
    lote = con_salida.unionByName(sin_salida)

    (DeltaTable.forName(lote.sparkSession, qualified_name(BRONZE)).alias("t")
        .merge(lote.alias("s"), """
            t.icao24 = s.icao24 AND (
                 (s.firstSeen IS NOT NULL AND t.firstSeen = s.firstSeen)
              OR (s.firstSeen IS NULL AND t.firstSeen IS NULL AND t.lastSeen = s.lastSeen)
            )""")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


def autoloader_write(stream_df):
    """ESCRITURA · cada lote del stream entra a bronze con MERGE; espera a que termine."""
    query = (stream_df.writeStream
             .foreachBatch(upsert_to_bronze)
             .option("checkpointLocation", CHECKPOINT_PATH)
             .trigger(availableNow=True)                              # procesa lo pendiente y se detiene
             .start())
    query.awaitTermination()


def run_bronze():
    """RAW → bronze: lectura + escritura (upsert) con Auto Loader."""
    create_bronze_table()
    antes = table_version(BRONZE)
    autoloader_write(autoloader_read())
    merges = (spark.sql(f"DESCRIBE HISTORY {BRONZE}")
              .where(f"version > {antes} AND operation = 'MERGE'")
              .select("operationMetrics").collect())
    if not merges:
        print("bronze: sin archivos nuevos")
        return
    nuevos = sum(int(m[0].get("numTargetRowsInserted", 0)) for m in merges)
    actualizados = sum(int(m[0].get("numTargetRowsUpdated", 0)) for m in merges)
    print(f"bronze: {nuevos} vuelos nuevos, {actualizados} actualizados (vistos en otro aeropuerto)")

## 6. Change Data Feed: leer cambios y recordar hasta dónde se procesó

In [0]:
def read_changes(tabla, desde, hasta):
    """Lee el Change Data Feed de una tabla entre dos versiones (ambas incluidas)."""
    return (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", desde)
            .option("endingVersion", hasta)
            .table(tabla))


def last_processed(step):
    """Hasta qué versión del origen ya procesó este paso (-1 si nunca corrió)."""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {STATE} (step STRING, last_version BIGINT, updated_at TIMESTAMP)")
    fila = spark.table(STATE).where(F.col("step") == step).first()
    return fila["last_version"] if fila else -1


def save_processed(step, version):
    """Guarda hasta qué versión procesó este paso."""
    spark.sql(f"""
        MERGE INTO {STATE} t
        USING (SELECT '{step}' AS step, {version} AS last_version, current_timestamp() AS updated_at) s
        ON t.step = s.step
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## 7. Silver: bronze → silver con CDF + MERGE

In [0]:
SILVER_COLS = ["vuelo_id", "icao24", "callsign", "aerolinea_icao", "origen_icao", "destino_icao",
               "fecha", "salida_local", "llegada_local", "duracion_min", "source_file"]


def create_silver_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER} (
            vuelo_id STRING, icao24 STRING, callsign STRING, aerolinea_icao STRING,
            origen_icao STRING, destino_icao STRING, fecha DATE,
            salida_local TIMESTAMP, llegada_local TIMESTAMP, duracion_min DOUBLE, source_file STRING
        ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)


def to_silver(df):
    """Una fila por vuelo: id único, hora de Brasil, aerolínea extraída del callsign, duración."""
    extra = [c for c in ("_change_type", "_commit_version") if c in df.columns]
    callsign = F.upper(F.trim(F.col("callsign")))
    salida   = F.from_utc_timestamp(F.timestamp_seconds("firstSeen"), ZONA_HORARIA)
    return (df.select(
        F.concat_ws("_", "icao24", F.col("firstSeen").cast("string")).alias("vuelo_id"),
        "icao24",
        F.when(callsign == "", None).otherwise(callsign).alias("callsign"),
        F.when(callsign.rlike("^[A-Z]{3}[0-9]"), callsign.substr(1, 3))
         .otherwise(F.lit("OTROS")).alias("aerolinea_icao"),          # privados o sin callsign
        F.coalesce(F.col("estDepartureAirport"), F.lit("DESC")).alias("origen_icao"),
        F.coalesce(F.col("estArrivalAirport"),   F.lit("DESC")).alias("destino_icao"),
        F.to_date(salida).alias("fecha"),
        salida.alias("salida_local"),
        F.from_utc_timestamp(F.timestamp_seconds("lastSeen"), ZONA_HORARIA).alias("llegada_local"),
        F.round((F.col("lastSeen") - F.col("firstSeen")) / 60, 1).alias("duracion_min"),
        "source_file", *extra)
      .where("icao24 IS NOT NULL AND duracion_min > 0 AND duracion_min <= 1200"))  


def build_silver():
    """bronze → silver: aplica solo los cambios de bronze que aún no se procesaron."""
    desde, hasta = last_processed("silver") + 1, table_version(BRONZE)
    if desde > hasta:
        print("silver al día: no hay cambios nuevos en bronze")
        return

    # 1. Cambios pendientes de bronze (la imagen previa de un UPDATE no se aplica)
    cambios = read_changes(BRONZE, desde, hasta).where("_change_type <> 'update_preimage'")

    # 2. Transformar y quedarse con el último cambio de cada vuelo
    ultimo = Window.partitionBy("vuelo_id").orderBy(F.col("_commit_version").desc())
    filas = (to_silver(cambios)
             .withColumn("rn", F.row_number().over(ultimo)).where("rn = 1").drop("rn"))

    # 3. MERGE: delete borra, update actualiza, insert inserta
    create_silver_table()
    valores = {c: f"s.{c}" for c in SILVER_COLS}
    (DeltaTable.forName(spark, qualified_name(SILVER)).alias("t")
        .merge(filas.alias("s"), "t.vuelo_id = s.vuelo_id")
        .whenMatchedDelete(condition="s._change_type = 'delete'")
        .whenMatchedUpdate(set=valores)
        .whenNotMatchedInsert(condition="s._change_type <> 'delete'", values=valores)
        .execute())

    # 4. Recordar hasta dónde llegamos
    save_processed("silver", hasta)
    print(f"silver: aplicados {filas.count()} cambios de bronze (versiones {desde} a {hasta})")

## 8. Gold: silver → gold con CDF + MERGE

In [0]:
FACT_COLS = ["vuelo_id", "fecha_key", "hora_salida", "aerolinea_icao", "origen_icao", "destino_icao",
             "salida_local", "llegada_local", "duracion_min"]


def create_fact_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {FACT} (
            vuelo_id STRING, fecha_key INT, hora_salida INT, aerolinea_icao STRING,
            origen_icao STRING, destino_icao STRING, salida_local TIMESTAMP,
            llegada_local TIMESTAMP, duracion_min DOUBLE
        )
    """)


def build_fact():
    """silver → gold_fact_vuelo con CDF + MERGE: solo los vuelos que cambiaron."""
    desde, hasta = last_processed("fact") + 1, table_version(SILVER)
    if desde > hasta:
        print("fact al día: no hay cambios nuevos en silver")
        return

    ultimo = Window.partitionBy("vuelo_id").orderBy(F.col("_commit_version").desc())
    filas = (read_changes(SILVER, desde, hasta)
             .where("_change_type <> 'update_preimage'")
             .withColumn("rn", F.row_number().over(ultimo)).where("rn = 1")
             .select("vuelo_id",
                     F.date_format("fecha", "yyyyMMdd").cast("int").alias("fecha_key"),
                     F.hour("salida_local").alias("hora_salida"),
                     "aerolinea_icao", "origen_icao", "destino_icao",
                     "salida_local", "llegada_local", "duracion_min", "_change_type"))

    create_fact_table()
    valores = {c: f"s.{c}" for c in FACT_COLS}
    (DeltaTable.forName(spark, qualified_name(FACT)).alias("t")
        .merge(filas.alias("s"), "t.vuelo_id = s.vuelo_id")
        .whenMatchedDelete(condition="s._change_type = 'delete'")
        .whenMatchedUpdate(set=valores)
        .whenNotMatchedInsert(condition="s._change_type <> 'delete'", values=valores)
        .execute())

    save_processed("fact", hasta)
    print(f"fact: aplicados {filas.count()} cambios de silver (versiones {desde} a {hasta})")


def build_dims():
    """Dimensiones: catálogos (CSV) + todos los códigos que aparecen en silver, para que ninguna FK quede huérfana."""
    s = spark.table(SILVER)

    # ---- dim_aerolinea ----
    cat_aerolineas = spark.read.option("header", True).csv(f"{RAW_CATALOGOS}/aerolineas.csv")
    (s.select("aerolinea_icao").distinct()
      .join(cat_aerolineas, "aerolinea_icao", "left")
      .fillna({"aerolinea": "Sin catálogo", "pais": "Sin catálogo", "tipo": "Sin catálogo"})
      .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(DIM_AEROLINEA))

    # ---- dim_aeropuerto ----
    aeropuertos = (spark.read.option("header", True).csv(f"{RAW_CATALOGOS}/airports.csv")
                   .select(F.col("ident").alias("aeropuerto_icao"),
                           F.col("name").alias("aeropuerto"),
                           F.col("municipality").alias("ciudad"),
                           "iso_country", "iata_code",
                           F.col("latitude_deg").cast("double").alias("latitud"),
                           F.col("longitude_deg").cast("double").alias("longitud")))
    paises = (spark.read.option("header", True).csv(f"{RAW_CATALOGOS}/countries.csv")
              .select(F.col("code").alias("iso_country"), F.col("name").alias("pais")))
    codigos = (s.select(F.col("origen_icao").alias("aeropuerto_icao"))
                .union(s.select("destino_icao")).distinct())

    (codigos.join(aeropuertos, "aeropuerto_icao", "left")
            .join(paises, "iso_country", "left")
            .withColumn("ambito",
                F.when(F.col("aeropuerto_icao") == "DESC", "Desconocido")
                 .when(F.col("iso_country").isNull(), "Sin catálogo")
                 .when(F.col("iso_country") == "BR", "Nacional")
                 .otherwise("Internacional"))
            .withColumn("es_aeropuerto_estudiado", F.col("aeropuerto_icao").isin(AEROPUERTOS))
            .withColumn("aeropuerto", F.when(F.col("aeropuerto_icao") == "DESC", "Desconocido")
                                       .otherwise(F.coalesce("aeropuerto", F.lit("Sin catálogo"))))
            .fillna({"ciudad": "Sin catálogo", "pais": "Sin catálogo"})
            .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(DIM_AEROPUERTO))

    # ---- dim_fecha (calendario en español) ----
    spark.sql(f"""
        CREATE OR REPLACE TABLE {DIM_FECHA} AS
        SELECT CAST(date_format(d, 'yyyyMMdd') AS INT)                        AS fecha_key,
               d                                                              AS fecha,
               year(d)                                                        AS anio,
               month(d)                                                       AS mes,
               element_at(array('Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio',
                                'Agosto','Setiembre','Octubre','Noviembre','Diciembre'), month(d)) AS nombre_mes,
               dayofweek(d)                                                   AS dia_semana_num,
               element_at(array('Domingo','Lunes','Martes','Miércoles','Jueves',
                                'Viernes','Sábado'), dayofweek(d))            AS dia_semana,
               dayofweek(d) IN (1, 7)                                         AS es_fin_de_semana
        FROM (SELECT explode(sequence(DATE'2026-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS d)
    """)
    print("dims: aerolínea, aeropuerto y fecha listas")


def build_gold():
    """silver → gold: dimensiones (completas) + hechos (incrementales con CDF)."""
    build_dims()
    build_fact()

## 9. Reiniciar el pipeline

Borra las tablas y la memoria de Auto Loader. **RAW no se toca** (los JSON descargados se conservan).

In [0]:
def reset_pipeline():
    """Deja el pipeline como nuevo: tablas y memoria de Auto Loader. RAW se conserva."""
    for t in (BRONZE, SILVER, FACT, DIM_AEROLINEA, DIM_AEROPUERTO, DIM_FECHA, STATE):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    for p in (SCHEMA_PATH, CHECKPOINT_PATH):
        dbutils.fs.rm(p, recurse=True)
    print("✔ Pipeline reiniciado (RAW intacto)")

In [0]:
def validate_silver(min_filas=1):
    """Reglas de calidad de silver. Si una falla, lanza un error y el Job se detiene antes de publicar."""
    s = spark.table(SILVER)
    filas = s.count()
    reglas = {
        f"hay al menos {min_filas} vuelos":   filas >= min_filas,
        "vuelo_id sin nulos":                 s.where("vuelo_id IS NULL").count() == 0,
        "vuelo_id sin duplicados":            s.select("vuelo_id").distinct().count() == filas,
        "fecha sin nulos":                    s.where("fecha IS NULL").count() == 0,
        "duración entre 0 y 20 h":            s.where("duracion_min <= 0 OR duracion_min > 1200").count() == 0,
        "llegada posterior a la salida":      s.where("llegada_local <= salida_local").count() == 0,
    }
    for regla, ok in reglas.items():
        print("✔" if ok else "✘", regla)
    fallidas = [r for r, ok in reglas.items() if not ok]
    if fallidas:
        raise Exception(f"Validación fallida: {fallidas}")
    print(f"silver válida: {filas} vuelos")


def log_run(estado, detalle=""):
    """Registra la corrida del pipeline en una tabla Delta (log del proceso)."""
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {LOG} (
        ejecutado_en TIMESTAMP, estado STRING, archivos_raw INT,
        filas_bronze BIGINT, filas_silver BIGINT, filas_fact BIGINT, detalle STRING)""")
    fila = [(estado, len(raw_files()),
             spark.table(BRONZE).count(), spark.table(SILVER).count(),
             spark.table(FACT).count(), detalle)]
    (spark.createDataFrame(fila, "estado STRING, archivos_raw INT, filas_bronze BIGINT, "
                                 "filas_silver BIGINT, filas_fact BIGINT, detalle STRING")
          .withColumn("ejecutado_en", F.current_timestamp())
          .select("ejecutado_en", "estado", "archivos_raw", "filas_bronze", "filas_silver", "filas_fact", "detalle")
          .write.mode("append").saveAsTable(LOG))

## 10. Ayudantes del Job

In [0]:
w = WorkspaceClient()    
def find_job(name):
    """Id del Job con ese nombre (None si no existe)."""
    jobs = w.api_client.do("GET", "/api/2.2/jobs/list", query={"name": name}).get("jobs", [])
    return jobs[0]["job_id"] if jobs else None


def create_job(settings):
    """Crea el Job; si ya existe uno con ese nombre, lo actualiza. Devuelve su id."""
    job_id = find_job(settings["name"])
    if job_id:
        w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": job_id, "new_settings": settings})
    else:
        job_id = w.api_client.do("POST", "/api/2.2/jobs/create", body=settings)["job_id"]
    print(f"Job listo: {settings['name']} → {w.config.host}/jobs/{job_id}")
    return job_id


def run_job(job_id, **params):
    """Ejecuta el Job con esos parámetros, espera a que termine y muestra el estado de cada tarea."""
    run_id = w.api_client.do("POST", "/api/2.2/jobs/run-now",
                             body={"job_id": job_id, "job_parameters": params})["run_id"]
    print(f"Corrida en marcha: {w.config.host}/jobs/{job_id}/runs/{run_id}")
    while True:
        run = w.api_client.do("GET", "/api/2.2/jobs/runs/get", query={"run_id": run_id})
        if run["state"]["life_cycle_state"] in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            break
        time.sleep(15)
    estados = {}
    for t in sorted(run["tasks"], key=lambda t: (t.get("start_time") or float("inf"), t.get("attempt_number", 0))):
        estados[t["task_key"]] = t["state"].get("result_state", t["state"]["life_cycle_state"])
    for tarea, estado in estados.items():
        print(f"  {tarea:<15} {estado}")
    print("Resultado del Job:", run["state"].get("result_state", run["state"]["life_cycle_state"]))

In [0]:
# Columnas por las que más se filtra cada tabla → ZORDER BY
ZORDER_COLS = {
    BRONZE: ["icao24", "firstSeen"],          # clave del MERGE de bronze
    SILVER: ["fecha", "aerolinea_icao"],      # filtros de build_fact y de la validación
    FACT:   ["fecha_key", "aerolinea_icao"],  # filtros más usados en Power BI
}


def mantenimiento(retencion_horas=168):
    """OPTIMIZE + ZORDER BY y VACUUM sobre las tablas del pipeline.
    Retención de 168 h (7 días): conserva el time travel y las versiones que lee el CDF."""
    for tabla, cols in ZORDER_COLS.items():
        m = spark.sql(f"OPTIMIZE {tabla} ZORDER BY ({', '.join(cols)})").first()["metrics"]
        print(f"OPTIMIZE {tabla:<16} ZORDER BY {cols} → "
              f"archivos quitados: {m['numFilesRemoved']}, agregados: {m['numFilesAdded']}")
        spark.sql(f"VACUUM {tabla} RETAIN {retencion_horas} HOURS")
        print(f"VACUUM   {tabla:<16} RETAIN {retencion_horas} HOURS ✔")

In [0]:
print("✔ Setup listo")
print(f"  Esquema : {CATALOG}.{SCHEMA}")
print(f"  Archivos: {BASE}")
print(f"  Tablas  : {BRONZE} → {SILVER} → {FACT} (+ {DIM_AEROLINEA}, {DIM_AEROPUERTO}, {DIM_FECHA})")
print(f"  Log     : {LOG}")